<a href="https://colab.research.google.com/github/mugalan/introduction-to-statistical-learning/blob/main/assignments/Bayesian_Inference_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Q Bayesian Estimations for Structural Health Monitoring via Bounded Grid Updates

In aerospace and civil engineering, Structural Health Monitoring (SHM) is critical for detecting damage before a catastrophic failure occurs. Consider an aircraft wing or a bridge girder equipped with specialized vibration sensors. Over time, environmental fatigue or dynamic impacts can cause micro-fractures, resulting in a reduction of the component's mechanical stiffness.

Let $\Theta = \theta$ represent the structural **remaining stiffness efficiency factor**, where $\theta$ is physically bounded to the interval:

$$\theta \in (0, 1]$$

* $\theta = 1.0$ indicates a perfectly pristine, undamaged structural component.
* $\theta \to 0$ signifies critical degradation or severe structural cracking.

Let $K_{\text{nominal}}$ be the known, baseline stiffness of the structural component when it is entirely healthy. At each sequential inspection time step $k$ (where $k = 1, 2, \dots, n$), a sensor collects a noisy experimental stiffness measurement $y_k$.

Engineers model the degradation physics via a non-linear relationship with multiplicative log-normal measurement noise to prevent non-physical negative values:

$$y_k = \theta \cdot K_{\text{nominal}} \cdot e^{\epsilon_k}, \qquad \epsilon_k \sim \mathscr{N}(0, \sigma^2)$$

where $\sigma$ is the standard deviation of the sensor noise in log-space.

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running history vector of observed sensor readings** up to the current inspection milestone. Before deploying the sensors, engineers utilize an initial prior distribution $f_{\Theta}^{(0)}(\theta)$ over the domain $(0, 1]$ based on historical manufacturing specifications. As the sensor stream arrives, the posterior distribution calculated at step $k-1$ serves directly as the prior distribution for step $k$.

---

### **Tasks**

#### **1. Prior Belief Boundaries**

Before data collection begins, engineers assume the component is highly likely to be healthy, modeling this using a bounded Beta distribution as the initial prior: $\Theta \sim \text{Beta}(8, 1.5)$.

* Plot this initial prior density function using Plotly over the restricted physical domain $\theta \in [0.01, 1.0]$.
* Calculate the expected prior stiffness efficiency $\mathbb{E}[\Theta^{(0)}]$ analytically. Explain why this specific distribution serves as an appropriate initial prior for an engineering component assumed to be healthy.

#### **2. Structural Likelihood Formulation**

Using the change of variables or properties of the log-normal distribution, write down the mathematical likelihood contribution $L(y_k \mid \theta)$ of a *single* continuous sensor measurement $y_k$ at inspection step $k$, given the true stiffness factor $\theta$. Following this, write down the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.

#### **3. Mathematical Formulation of the Non-Conjugate Grid Update**

Explain why an exact closed-form analytical solution for the posterior density $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$ does not exist when combining a Beta prior with this log-normal structural likelihood. Write down the recursive relationship for the posterior density at step $k$ up to a proportionality constant.

#### **4. Running Point Estimates**

Because a closed-form formula is unavailable, we must define point estimators through numerical integration. Write down the definite integral equations over the bounded domain $(0, 1]$ required to compute:

* The **Running Posterior Mean** ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$)
* The **Running Maximum A Posteriori** ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$)

#### **5. Algorithmic Grid Approximation and Normalization**

Describe the step-by-step numerical procedure to maintain this distribution on a discrete grid of $\theta$-values. Explicitly state how you would handle the boundary limits computationally and how you would perform the sequential normalization step using the trapezoidal rule after a new sensor reading $y_k$ is observed.

#### **6. Performance Tracking and Degradation Convergence Analysis**

Suppose an impact occurs, and the true, hidden remaining stiffness drops to $\theta_{\text{true}} = 0.68$. Write a Python script using Plotly to simulate an engineered monitoring timeline across $n = 15$ continuous sensor measurements ($K_{\text{nominal}} = 50.0 \text{ kN/mm}$, $\sigma = 0.15$):

* **Simulate Sensor Stream:** Programmatically generate noisy sensor readings $y_k$ by drawing random values from the underlying log-normal physics model centered at $\theta_{\text{true}}$.
* **Track Estimators:** Loop sequentially through each step. At each step, update the unnormalized grid, normalize it via `np.trapezoid`, and compute both $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ and $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$.
* **Visualize Curves & Timeline:** Generate two plots:
1. A plot showing the progression of the full posterior density curves at milestones $k \in \{0, 1, 2, 5, 10, 15\}$.
2. A line chart tracking the convergence of both $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ and $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$ from step $0$ to $15$ against a horizontal reference line at $\theta_{\text{true}} = 0.68$.


* **Analysis:** Evaluate the behavior of the distribution. How many sensor readings did it take for the system to overcome the initially optimistic "healthy" prior and confidently isolate the 68% damage state? What does the narrowing of the density curves imply about structural safety thresholds?

## 1. Prior Belief Boundaries

The expected prior stiffness efficiency $\mathbb{E}[\Theta^{(0)}]$ for a Beta distribution is given by the formula $\frac{\alpha}{\alpha + \beta}$:

$$\mathbb{E}[\Theta^{(0)}] = \frac{8}{8 + 1.5} = \frac{8}{9.5} \approx 0.842$$

This distribution is highly appropriate for an initial prior because it assigns the vast majority of the probability mass near $1.0$ (representing an undamaged or lightly degraded state), which aligns with the physical reality of deploying a healthy component. The density gently tapers off toward the lower values, leaving a small non-zero probability for pre-existing manufacturing defects or undocumented damage without letting those scenarios dominate the baseline assumption.

In [5]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta

theta_grid = np.linspace(0.01, 1.0, 500)
prior_density = beta.pdf(theta_grid, 8, 1.5)

fig = go.Figure()
fig.add_trace(go.Scatter(x=theta_grid, y=prior_density, mode='lines', name='Beta(8, 1.5) Prior'))
fig.update_layout(title='Initial Prior Belief of Stiffness Efficiency',
                  xaxis_title='Stiffness Efficiency (theta)',
                  yaxis_title='Density',
                  template='plotly_white')
fig.show()



## 2. Structural Likelihood Formulation

Taking the natural logarithm of the physical measurement model $y_k = \theta \cdot K_{\text{nominal}} \cdot e^{\epsilon_k}$ gives:

$$\ln(y_k) = \ln(\theta \cdot K_{\text{nominal}}) + \epsilon_k$$

Since $\epsilon_k \sim \mathscr{N}(0, \sigma^2)$, the variable $\ln(y_k)$ is normally distributed with mean $\mu = \ln(\theta K_{\text{nominal}})$ and variance $\sigma^2$. Therefore, $y_k$ follows a log-normal distribution. The likelihood contribution of a single measurement $y_k$ is:

$$L(y_k \mid \theta) = \frac{1}{y_k \sigma \sqrt{2\pi}} \exp\left( -\frac{[\ln(y_k) - \ln(\theta K_{\text{nominal}})]^2}{2\sigma^2} \right)$$

Assuming the sensor measurements are conditionally independent given $\theta$, the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$ is the product of the individual likelihoods:

$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{j=1}^{k} \frac{1}{y_j \sigma \sqrt{2\pi}} \exp\left( -\frac{[\ln(y_j) - \ln(\theta K_{\text{nominal}})]^2}{2\sigma^2} \right)$$

## 3. Mathematical Formulation of the Non-Conjugate Grid Update

An exact closed-form analytical solution does not exist because the Beta prior (characterized by a polynomial form $\theta^{\alpha-1}(1-\theta)^{\beta-1}$) and the log-normal likelihood (characterized by an exponential term $\exp(-(\ln \theta + C)^2)$) are not conjugate. When multiplied, they do not algebraically simplify into a recognizable, standard probability distribution family.

Instead, the recursive relationship must be evaluated numerically up to a proportionality constant:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto L(y_k \mid \theta) \times f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$$

## 4. Running Point Estimates

Since we cannot express the posterior function analytically, the point estimators must be defined using formal integration and maximization over the bounded domain $(0, 1]$:

*   **Running Posterior Mean:**
    $$\widehat{\theta}_{\mathrm{Bayes}}^{(k)} = \int_{0}^{1} \theta \cdot f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \, d\theta$$

*   **Running Maximum A Posteriori (MAP):**
    $$\widehat{\theta}_{\mathrm{MAP}}^{(k)} = \underset{\theta \in (0, 1]}{\operatorname{arg\,max}} \, f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$$

## 5. Algorithmic Grid Approximation and Normalization

To handle this non-conjugate update computationally:

1.  **Grid Initialization:** Define a discrete, finely spaced one-dimensional array corresponding to $\theta$ from a small value $\epsilon$ (e.g., 0.01) to 1.0. We strictly avoid $0.0$ to prevent undefined $\ln(0)$ evaluations in the likelihood function.
2.  **Prior Array:** Evaluate the initial Beta(8, 1.5) density at each point on the grid to create the starting posterior array.
3.  **Sequential Update:** For each new measurement $y_k$:
    *   Evaluate the log-normal likelihood formula $L(y_k \mid \theta)$ across the entire grid.
    *   Perform element-wise multiplication of the likelihood array and the current posterior array to yield the unnormalized posterior array.
    *   **Normalization:** Compute the definite integral of the unnormalized array using the trapezoidal rule. Divide the unnormalized posterior array by this scalar area value so the density integrates precisely to 1.0.
4.  **Extract Estimates:** Compute the Posterior Mean by numerically integrating $\theta \cdot f(\theta)$ via the trapezoidal rule, and the MAP by identifying the $\theta$ value at the maximum array index.

## 6. Performance Tracking and Degradation Convergence Analysis

### Analysis of the System

The progression shows that it takes approximately 3 to 5 sensor readings for the likelihood functions to overpower the heavily skewed, optimistic initial prior. By step 5, the density has fundamentally shifted away from 0.84 and centered near the true state of 0.68.

As $k$ approaches 15, the posterior density curves become progressively narrower (reduced variance) and taller (higher peak density). This narrowing implies that the system is growing increasingly confident in its estimation of the damage state. In an engineering context, this sharp certainty is critical—tight confidence intervals prevent false-positive alarms while ensuring that safety threshold decisions (e.g., grounding an aircraft if the 95% credible interval drops below $\theta=0.70$) are driven by hard, statistical evidence rather than baseline assumptions.

In [6]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta
from plotly.subplots import make_subplots

# 1. Simulation Parameters
np.random.seed(42)
n_steps = 15
theta_true = 0.68
K_nominal = 50.0
sigma = 0.15

# Generate noisy sensor readings
epsilon = np.random.normal(0, sigma, n_steps)
y_measurements = theta_true * K_nominal * np.exp(epsilon)

# 2. Grid Setup
grid_points = 1000
theta_grid = np.linspace(0.01, 1.0, grid_points)

# Trackers
posterior = beta.pdf(theta_grid, 8, 1.5)
posteriors_to_plot = {0: posterior.copy()}
mean_estimates = [np.trapezoid(theta_grid * posterior, x=theta_grid)]
map_estimates = [theta_grid[np.argmax(posterior)]]

# 3. Sequential Update Loop
for k, y_k in enumerate(y_measurements, start=1):
    # Calculate Likelihood
    likelihood = (1 / (y_k * sigma * np.sqrt(2 * np.pi))) * \
                 np.exp(- (np.log(y_k) - np.log(theta_grid * K_nominal))**2 / (2 * sigma**2))
    
    # Update and Normalize
    unnormalized = posterior * likelihood
    normalization_constant = np.trapezoid(unnormalized, x=theta_grid)
    posterior = unnormalized / normalization_constant
    
    # Store selected posteriors for visualization
    if k in [1, 2, 5, 10, 15]:
        posteriors_to_plot[k] = posterior.copy()
        
    # Point Estimates
    mean_est = np.trapezoid(theta_grid * posterior, x=theta_grid)
    map_est = theta_grid[np.argmax(posterior)]
    
    mean_estimates.append(mean_est)
    map_estimates.append(map_est)

# 4. Visualization
fig = make_subplots(rows=1, cols=2, subplot_titles=("Evolution of Posterior Density", "Convergence of Estimators"),
                    horizontal_spacing=0.15)

# Plot 1: Posterior densities
colors = ['#c8d6e5', '#8395a7', '#576574', '#222f3e', '#10ac84', '#ee5253']
for idx, (k, post_density) in enumerate(posteriors_to_plot.items()):
    fig.add_trace(go.Scatter(x=theta_grid, y=post_density, mode='lines', 
                             name=f'Step {k}', line=dict(color=colors[idx], width=2)), row=1, col=1)

# Plot 2: Convergence
steps = list(range(n_steps + 1))
fig.add_trace(go.Scatter(x=steps, y=mean_estimates, mode='lines+markers', name='Posterior Mean'), row=1, col=2)
fig.add_trace(go.Scatter(x=steps, y=map_estimates, mode='lines+markers', name='MAP Estimate'), row=1, col=2)
fig.add_trace(go.Scatter(x=steps, y=[theta_true]*(n_steps+1), mode='lines', 
                         name='True Theta (0.68)', line=dict(dash='dash', color='red')), row=1, col=2)

fig.update_layout(template='plotly_white', height=500, title_text="Structural Health Monitoring: Bayesian Updating")
fig.update_xaxes(title_text="Stiffness Efficiency (theta)", row=1, col=1)
fig.update_yaxes(title_text="Density", row=1, col=1)
fig.update_xaxes(title_text="Inspection Step (k)", row=1, col=2)
fig.update_yaxes(title_text="Estimated Efficiency", row=1, col=2)
fig.show()